# Volue actuals ingestion (bronze → silver → gold)

Ingests Volue actuals curve `con {region} unscaled mwh/h cet min15 a` into `ccm_dev` schemas (bronze, silver, gold). Assumes secrets `VOLUE_CLIENT_ID` and `VOLUE_CLIENT_SECRET` are stored in a Databricks secret scope. Timestamps are normalized to UTC; Delta is used throughout.

In [ ]:
%pip install volue-insight-timeseries

In [ ]:
# Widgets for runtime overrides (Databricks jobs can pass these)
if "region" not in dbutils.widgets.getArgumentNames():
    dbutils.widgets.text("region", "SE3")
if "lookback_days" not in dbutils.widgets.getArgumentNames():
    dbutils.widgets.text("lookback_days", "2")
if "secret_scope" not in dbutils.widgets.getArgumentNames():
    dbutils.widgets.text("secret_scope", "volue-secrets")
if "client_id_key" not in dbutils.widgets.getArgumentNames():
    dbutils.widgets.text("client_id_key", "VOLUE_CLIENT_ID")
if "client_secret_key" not in dbutils.widgets.getArgumentNames():
    dbutils.widgets.text("client_secret_key", "VOLUE_CLIENT_SECRET")

region = dbutils.widgets.get("region")
lookback_days = int(dbutils.widgets.get("lookback_days"))
secret_scope = dbutils.widgets.get("secret_scope")
client_id_key = dbutils.widgets.get("client_id_key")
client_secret_key = dbutils.widgets.get("client_secret_key")

bronze_db = "ccm_dev.mfrr_forecast_bronze"
silver_db = "ccm_dev.mfrr_forecast_silver"
gold_db = "ccm_dev.mfrr_forecast_gold"
bronze_table = f"{bronze_db}.volue_actuals"
silver_table = f"{silver_db}.volue_actuals_clean"
gold_view = f"{gold_db}.volue_actuals_latest"
watermark_table = f"{bronze_db}.volue_actuals_watermark"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {bronze_db}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {silver_db}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {gold_db}")

In [ ]:
import pandas as pd
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as F, types as T
import volue_insight_timeseries

client_id = dbutils.secrets.get(secret_scope, client_id_key)
client_secret = dbutils.secrets.get(secret_scope, client_secret_key)
session = volue_insight_timeseries.Session(client_id=client_id, client_secret=client_secret)
curve_name = f"con {region} unscaled mwh/h cet min15 a"

print(f"Using curve: {curve_name}")

In [ ]:
def load_watermark() -> datetime:
    if not spark.catalog.tableExists(watermark_table):
        return datetime.now(timezone.utc) - timedelta(days=lookback_days)
    wm_row = (spark.table(watermark_table)
        .filter(F.col("curve_id") == curve_name)
        .agg(F.max("last_timestamp_utc").alias("wm"))
        .first())
    if wm_row is None or wm_row.wm is None:
        return datetime.now(timezone.utc) - timedelta(days=lookback_days)
    return max(wm_row.wm.replace(tzinfo=timezone.utc), datetime.now(timezone.utc) - timedelta(days=lookback_days))

watermark = load_watermark()
data_from = watermark
data_to = datetime.now(timezone.utc)
print(f"Pulling window: {data_from.isoformat()} → {data_to.isoformat()}")

In [ ]:
curve = session.get_curve(name=curve_name)
ts = curve.get_data(data_from=data_from.isoformat(), data_to=data_to.isoformat())
s = ts.to_pandas()
if getattr(s.index, "tz", None) is None:
    s.index = s.index.tz_localize("Europe/Oslo")
s.index = s.index.tz_convert("UTC")
df_pd = s.to_frame(name="value").reset_index().rename(columns={"index": "timestamp_utc"})
df_pd["curve_id"] = curve_name
df_pd["region"] = region
df_pd["unit"] = "mwh/h"
df_pd["issue_date"] = pd.NaT
df_pd["source_modified_at"] = pd.NaT
df_pd["ingested_at"] = pd.Timestamp.utcnow().tz_localize(None)
df_pd["source_system"] = "volue"
df_pd["date"] = df_pd["timestamp_utc"].dt.date

if df_pd.empty:
    raise ValueError("No data returned for requested window.")

schema = T.StructType([
bronze_df = spark.createDataFrame(df_pd, schema=schema)
bronze_df = bronze_df.select("curve_id", "region", "timestamp_utc", "value", "unit", "issue_date", "source_modified_at", "ingested_at", "source_system", "date")
bronze_df.display()

In [ ]:
spark.sql("SET spark.sql.legacy.timeParserPolicy=LEGACY")

if not spark.catalog.tableExists(bronze_table):
    (bronze_df
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("date")
        .saveAsTable(bronze_table))
else:
    bronze_df.createOrReplaceTempView("incoming_volue_actuals")
    spark.sql(f"

)

print(f"Upserted {bronze_df.count()} rows into {bronze_table}")

In [ ]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {watermark_table} (curve_id STRING, last_timestamp_utc TIMESTAMP, updated_at TIMESTAMP) USING delta")
max_ts = bronze_df.agg(F.max("timestamp_utc").alias("max_ts")).first().max_ts
if max_ts is not None:
    wm_df = spark.createDataFrame([(curve_name, max_ts, datetime.now(timezone.utc))], ["curve_id", "last_timestamp_utc", "updated_at"])
    wm_df.createOrReplaceTempView("incoming_wm")
    spark.sql(f"

)
print(f"Watermark updated to {max_ts}")

In [ ]:
silver_df = (spark.table(bronze_table)
(silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("date")
        .saveAsTable(silver_table))
print(f"Refreshed silver table {silver_table}")

In [ ]:
spark.sql(f"
60

)
print(f"Created/updated gold view {gold_view}")